## M1 - Problem Statement

Parse loan applications and financial statements into validated Pydantic LoanApplication / FinancialSnapshot objects via a provider-agnostic client.

In [35]:
import os, json
from typing import Literal, List
from litellm import completion
from pydantic import BaseModel, ValidationError

from dotenv import load_dotenv
load_dotenv(".env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    print("GEMINI_API_KEY is empty — paste your key above, between the quotes, then re-run this cell.")
else:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print("Environment OK — GEMINI_API_KEY found.")

Environment OK — GEMINI_API_KEY found.


In [12]:
## read the jsonl file - the loan application records

with open("data/wealthpilot/intake/records.jsonl") as f:
    records = [json.loads(line) for line in f]
print(f"Loaded {len(records)} records from records.jsonl.")

Loaded 20 records from records.jsonl.


In [20]:
EXTRACTION_PROMPT = """Extract loan application data from the text below into JSON with exactly these fields if present:
application_id (string or null), channel (string or Null), received_at (string or null),
raw_narrative (string or null), business name (string or null), sector (string), requested_amount_inr (number),
declared_financials (object with annual_revenue_inr, ebitda_inr, existing_debt_inr, current_assets_inr, current_liabilities_inr, months_operating),
bureau_score (number or null),
ground_truth (object with dscr, meets_dscr_floor, risk_band, requires_human_signoff, missing_fields (list of strings)),
record_id (string or null).
Return ONLY valid JSON, no markdown fences, no commentary.

Text:
{text}"""

def extract_raw(text: str) -> str:
    # Implement extraction from raw data
    resp = completion(
        #model="groq/llama-3.1-8b-instant",
        model = "gemini/gemini-flash-lite-latest",
        messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(text=text)}],
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

In [29]:
# Define Pydantic models for the extracted data

class DeclaredFinancials(BaseModel):
    annual_revenue_inr: int | None = None
    ebitda_inr: int
    existing_debt_inr: int
    current_assets_inr: int | None = None
    current_liabilities_inr: int | None = None
    months_operating: int | None = None

class GroundTruth(BaseModel):
    dscr: float | None = None
    meets_dscr_floor: bool | None = None
    risk_band: Literal["approve", "borderline", "decline"] | None = None
    requires_human_signoff: bool | None = None
    missing_fields: List[str] | None = None

class LoanApplication(BaseModel):
    application_id: str | None = None
    channel: str | None = None
    received_at: str | None = None
    raw_narrative: str | None = None
    business_name: str | None = None
    sector: str | None = None
    requested_amount_inr: float
    declared_financials: DeclaredFinancials
    bureau_score: float
    ground_truth: GroundTruth | None = None
    record_id: str | None = None

As the data itself in the JSON format. Skipping the LLM extraction step for now, we will focus on validating the data and converting it into Pydantic objects.

In [24]:
# Validate the records

def validate_loan_appl(raw: str) -> tuple[LoanApplication | None, str | None]:
    try:
        if isinstance(raw, str):
            raw = json.loads(raw)
        loan_application = LoanApplication(**raw)
        return loan_application, None
    except (json.JSONDecodeError, ValidationError) as e:
        return None, str(e)

In [25]:
results = []

for i in range(len(records)):
    loan_application, error = validate_loan_appl(records[i])
    results.append({"id":i, "loan_application": loan_application, "errors": error})

success = sum(1 for r in results if r["loan_application"] is not None)
print(f"Parsed {success}/{len(results)} records successfully.")


Parsed 15/20 records successfully.


In [26]:
## Listing down the records that failed validation, if any
[r for r in results if r["errors"] is not None]

[{'id': 2,
  'loan_application': None,
  'errors': '1 validation error for LoanApplication\nbureau_score\n  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]\n    For further information visit https://errors.pydantic.dev/2.13/v/float_type'},
 {'id': 5,
  'loan_application': None,
  'errors': "2 validation errors for LoanApplication\ndeclared_financials.ebitda_inr\n  Field required [type=missing, input_value={'annual_revenue_inr': 22... 'months_operating': 42}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nground_truth.dscr\n  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]\n    For further information visit https://errors.pydantic.dev/2.13/v/float_type"},
 {'id': 8,
  'loan_application': None,
  'errors': '1 validation error for LoanApplication\nbureau_score\n  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]\n    For fur

In [31]:
## Testing with a raw text input to see if the extraction and validation works end-to-end
text = "SME term loan request, 3,000,000 rupees. Textiles. 62 months operating. Revenue 24,000,000. EBITDA 3,120,000. Existing debt 1,900,000. Bureau score 741. Current ratio 1.4. Proprietor: Rajesh Sharma, Bengaluru."
raw = extract_raw(text)
print(raw)

{
  "application_id": null,
  "channel": null,
  "received_at": null,
  "raw_narrative": "SME term loan request, 3,000,000 rupees. Textiles. 62 months operating. Revenue 24,000,000. EBITDA 3,120,000. Existing debt 1,900,000. Bureau score 741. Current ratio 1.4. Proprietor: Rajesh Sharma, Bengaluru.",
  "business name": null,
  "sector": "Textiles",
  "requested_amount_inr": 3000000,
  "declared_financials": {
    "annual_revenue_inr": 24000000,
    "ebitda_inr": 3120000,
    "existing_debt_inr": 1900000,
    "current_assets_inr": null,
    "current_liabilities_inr": null,
    "months_operating": 62
  },
  "bureau_score": 741,
  "ground_truth": {
    "dscr": null,
    "meets_dscr_floor": null,
    "risk_band": null,
    "requires_human_signoff": null,
    "missing_fields": []
  },
  "record_id": null
}


In [32]:
loan_application, error = validate_loan_appl(raw)
if loan_application is not None:
    print("Validation successful.")
else:
    print(f"Validation failed with error: {error}")

Validation successful.
